# Data Exploration of different datasets

## Datasets used in this project

- [MARS dataset](https://www.sciencedirect.com/science/article/pii/S2352340923000604)
- [ITM-Rec dataset](https://arxiv.org/abs/2303.10230)

In [2]:
import ast
import re
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, OrdinalEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

In [ ]:
def sanitize_colname(prefix: str, col: str) -> str:
    """Creates a safe column name using regex."""
    s = str(col)
    clean_s = re.sub(r'[\s/\\\\]+', '_', s)
    return f"{prefix}__{clean_s}"

def parse_list_value(x):
    """Safely converts a value to a list."""
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try:
                parsed = ast.literal_eval(x)
                if isinstance(parsed, (list, tuple, set)):
                    return list(parsed)
            except (ValueError, SyntaxError):
                pass
    return [] if pd.isna(x) else [x]

def is_list_column(series: pd.Series, sample_size: int = 100) -> bool:
    """
    Checks if a column contains list-like objects based on a sample.
    Checking the entire series is often too slow.
    """
    sample = series.dropna().head(sample_size)
    if sample.empty:
        return False
    
    for x in sample:
        if isinstance(x, (list, tuple, set)):
            return True
        if isinstance(x, str) and x.strip().startswith("["):
            try:
                if isinstance(ast.literal_eval(x), (list, tuple, set)):
                    return True
            except Exception:
                continue
    return False

def process_list_columns(df: pd.DataFrame, list_cols: list[str]) -> pd.DataFrame:
    """Expands list columns using MultiLabelBinarizer and returns a DF with new cols."""
    encoded_frames = []
    
    for col in list_cols:
        normalized_data = [parse_list_value(x) for x in df[col]]
        
        mlb = MultiLabelBinarizer()
        encoded = mlb.fit_transform(normalized_data)
        
        bin_cols = [sanitize_colname(col, c) for c in mlb.classes_]
        
        df_mlb = pd.DataFrame(
            encoded,  # type: ignore
            columns=bin_cols, 
            index=df.index
        )
        encoded_frames.append(df_mlb)
        
    if not encoded_frames:
        return pd.DataFrame(index=df.index)
        
    return pd.concat(encoded_frames, axis=1)

def clean_text(text: str) -> str:
    """Limpieza básica: minúsculas y eliminación de caracteres no alfanuméricos."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess(df: pd.DataFrame, target: str, user_col: str, item_col: str, ignored_cols: list[str] = []) -> pd.DataFrame:
    """
    Optimized preprocessing pipeline.
    """
    df = df.copy()
    
    original_len = len(df)
    df = df.dropna(subset=[target])
    if len(df) < original_len:
        print(f"Dropped {original_len - len(df)} rows with NaN target.")

    le = LabelEncoder()
    df[user_col] = le.fit_transform(df[user_col].astype(str))
    df[item_col] = le.fit_transform(df[item_col].astype(str))

    protected_cols = set(ignored_cols + [user_col, item_col, target])
    candidates = [c for c in df.columns if c not in protected_cols]

    list_cols = []
    scalar_cols = []

    for col in candidates:
        if is_list_column(df[col]):
            list_cols.append(col)
        else:
            scalar_cols.append(col)

    dfs_to_concat = [df] # Start with the original
    
    if list_cols:
        print(f"Processing list columns: {list_cols}")
        df_lists_encoded = process_list_columns(df, list_cols)
        dfs_to_concat.append(df_lists_encoded)
        df = df.drop(columns=list_cols)
        dfs_to_concat[0] = df 

    for col in scalar_cols:
        if is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].mean()) # Buena práctica llenar NAs numéricos
            scaler = MinMaxScaler()
            df[col] = scaler.fit_transform(df[[col]])
        elif isinstance(df[col].dtype, pd.CategoricalDtype):
            series = df[col].astype(str).replace('nan', 'Undefined')
            oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            df[col] = oe.fit_transform(series.values.reshape(-1, 1))
        else:
            # TODO: Preprocess text data
            pass


    if len(dfs_to_concat) > 1:
        df_final = pd.concat(dfs_to_concat, axis=1)
    else:
        df_final = df

    return df_final

## MARS Dataset

In [4]:
explicit_df_en = pd.read_csv("../data/mars_dataset/explicit_ratings_en.csv")
explicit_df_fr = pd.read_csv("../data/mars_dataset/explicit_ratings_fr.csv")

items_en = pd.read_csv("../data/mars_dataset/items_en.csv")
items_fr = pd.read_csv("../data/mars_dataset/items_fr.csv")

df_explicit = pd.concat([explicit_df_en, explicit_df_fr], ignore_index=True)
df_items = pd.concat([items_en, items_fr], ignore_index=True)

df_explicit["created_at"] = pd.to_datetime(df_explicit["created_at"])
df_items = df_items.drop(columns=["created_at"])

df = pd.merge(df_explicit, df_items, on="item_id", how="inner")

df.rename(
    columns={"Difficulty": "difficulty", "type": "item_type", "Software": "software"},
    inplace=True,
)

df = df.astype({
    'difficulty': pd.CategoricalDtype(categories=['Beginner', 'Intermediate', 'Advanced'], ordered=True),
    'item_type': pd.CategoricalDtype(categories=['tutorial', 'video', 'software'], ordered=True),
    'language': pd.CategoricalDtype(categories=['en', 'fr'], ordered=True),
})

# features = [
#     "user_id",
#     "item_id",
#     "item_type",
#     "difficulty",
#     "nb_views",
#     "watch_percentage",
#     "description",
#     "rating",
# ]

# df = df[features]

print(df.dtypes)
df

user_id                      int64
item_id                      int64
watch_percentage             int64
created_at          datetime64[ns]
rating                       int64
language                  category
name                        object
nb_views                   float64
description                 object
difficulty                category
Job                         object
software                    object
Theme                       object
duration                   float64
item_type                 category
dtype: object


,user_id,item_id,watch_percentage,created_at,rating,language,name,nb_views,description,difficulty,Job,software,Theme,duration,item_type
0,224557,510,100,2018-09-28 16:18:29,10,en,What is OneDrive for Business?,1114.0,OneDrive for Businessis an online libraryto st...,Beginner,[],['OneDrive'],['Discover'],42.0,tutorial
1,224557,615,100,2018-09-28 16:22:22,10,en,Tell me what you want to do,184.0,Tell me brings featuresand helps topic to your...,Beginner,"['Accounting', 'Financial', 'Human resources',...",['Outlook'],"['Discover', 'Research']",57.0,tutorial
2,224557,7680,100,2018-09-28 16:23:34,10,en,Create a meeting in the group calendar,73.0,The Groups calendar helps youto track all the ...,Intermediate,[],['Outlook'],"['Organize', 'Collaborate']",72.0,tutorial
3,224293,510,100,2018-09-28 17:20:30,10,en,What is OneDrive for Business?,1114.0,OneDrive for Businessis an online libraryto st...,Beginner,[],['OneDrive'],['Discover'],42.0,tutorial
4,224293,515,100,2018-09-28 17:40:02,10,en,Work with documents in a synced library folder...,253.0,Once you sync your one drive library to your c...,Beginner,[],['OneDrive'],['Produce'],87.0,tutorial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88993,610452,419834,92,2021-09-21 15:14:13,10,fr,Présentation générale de Shift,42.0,NaN,NaN,[],['Shift'],['Découvrir'],83.0,tutorial
88994,610452,419835,100,2021-09-21 15:15:40,10,fr,Présentation de l'interface Shift,37.0,NaN,NaN,[],['Shift'],['Découvrir'],66.0,tutorial
88995,610452,419839,99,2021-09-21 15:16:58,10,fr,Qu'est-ce qu'un shift ouvert ?,33.0,NaN,NaN,[],['Shift'],['Découvrir'],40.0,tutorial
88996,610452,419841,100,2021-09-21 15:17:54,10,fr,Compléter le planning et le partager,37.0,NaN,NaN,[],['Shift'],"['Produire', 'Partager']",99.0,tutorial


In [5]:
df_preprocessed = preprocess(df, target="rating", user_col="user_id", item_col="item_id", ignored_cols=[])
df_preprocessed

Processing list columns: ['Job', 'software', 'Theme']


KeyboardInterrupt: 

In [56]:
num_nan_descriptions = df["description"].isna().sum()
print(f"Number of descriptions: {len(df['description'])}")
print(f"Number of descriptions with NaN: {num_nan_descriptions}")

Number of descriptions: 88998
Number of descriptions with NaN: 11203
